# MASAR 27 | مسار 27 — Final Demo Notebook

**AI-Powered Adaptive Travel Planner for AFC Asian Cup Saudi Arabia 2027 — Riyadh demo / multi-city concept**

هذه النسخة النهائية للديمو تشمل:
- تجربة تخطيط حوارية من أربع خطوات بدل استمارة طويلة.
- التخطيط قبل المباراة والمباراة وبعدها.
- Gemini Function Calling + Structured Output.
- محرك قيود حتمي للوقت، الصلاة، الحركة، موعد المباراة، والازدحام التقديري لكل محطة.
- إعادة تخطيط تكيفية عند الحرارة، الازدحام، التأخير، التعب، أو تغيّر وقت الصلاة.
- روابط Google Maps باسم المكان + Directions مباشر من الوجهة إلى الملعب.
- واجهة ثابتة على الهوية الفاتحة حتى لو كان جهاز المستخدم على Dark Mode.
- قراءة `MASAR27_DATA.xlsx` تلقائيًا من المشروع، مع نسخة بيانات مضمّنة احتياطية لضمان قابلية التشغيل.

> مفتاح Gemini لا يُكتب داخل الكود. ضعيه في Colab Secrets باسم `GEMINI_API_KEY`.
> النسخة الحالية نموذج الرياض فقط، بينما بنية الفكرة قابلة للتوسع لبقية المدن المستضيفة.


In [ ]:

# 1) تثبيت المكتبات
!pip install -q -U "google-genai>=2.13.0" "gradio>=6.0" pandas openpyxl requests pydantic


In [ ]:
# 2) تحميل الداتا + قراءة مفتاح Gemini بأمان من Colab Secrets
import os, re, json, math, html, requests, base64
from datetime import datetime, timedelta
import pandas as pd

DATA_FILENAME = "MASAR27_DATA.xlsx"

# بعد إنشاء GitHub يمكن وضع raw URL في متغير البيئة MASAR27_DATA_URL.
DATA_URL = os.getenv("MASAR27_DATA_URL", "").strip()

# Snapshot احتياطي من الداتا النهائية لضمان أن الديمو يعمل مباشرة.
# الملف المنفصل MASAR27_DATA.xlsx يظل المصدر الأساسي في حزمة المشروع/GitHub.
EMBEDDED_DATA_B64 = "..."

# ---------- API KEY: Colab Secrets ----------
GEMINI_API_KEY = ""
try:
    from google.colab import userdata
    GEMINI_API_KEY = (userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

# ---------- DATA FILE RESOLVER ----------
def _resolve_data_file():
    candidates = [
        f"/content/{DATA_FILENAME}",
        os.path.join(os.getcwd(), DATA_FILENAME),
        os.path.join(os.getcwd(), "data", DATA_FILENAME),
    ]
    for path in candidates:
        if os.path.exists(path):
            print(f"✅ تم العثور على الداتا تلقائيًا: {path}")
            return path

    if DATA_URL:
        target = f"/content/{DATA_FILENAME}"
        try:
            response = requests.get(DATA_URL, timeout=20)
            response.raise_for_status()
            with open(target, "wb") as f:
                f.write(response.content)
            print("✅ تم تنزيل الداتا تلقائيًا من رابط المشروع.")
            return target
        except Exception as e:
            print(f"⚠️ تعذر تنزيل الداتا من الرابط: {str(e)[:120]}")

    if EMBEDDED_DATA_B64:
        target = f"/content/{DATA_FILENAME}" if os.path.isdir("/content") else os.path.join(os.getcwd(), DATA_FILENAME)
        with open(target, "wb") as f:
            f.write(base64.b64decode(EMBEDDED_DATA_B64))
        print("✅ تم تجهيز نسخة الداتا المضمّنة تلقائيًا.")
        return target

    try:
        from google.colab import files
        print(f"📁 لم نجد الداتا تلقائيًا. ارفعي الملف: {DATA_FILENAME}")
        uploaded = files.upload()
        xlsx_files = [name for name in uploaded if name.lower().endswith(".xlsx")]
        if not xlsx_files:
            raise FileNotFoundError("لم يتم رفع ملف Excel.")
        return f"/content/{xlsx_files[0]}"
    except Exception as e:
        raise FileNotFoundError(f"تعذر الوصول إلى ملف الداتا: {e}")

DATA_FILE = _resolve_data_file()

xls = pd.ExcelFile(DATA_FILE)
df_matches = pd.read_excel(xls, sheet_name="Matches_All")
df_places = pd.read_excel(xls, sheet_name="Places_Riyadh")
df_sources = pd.read_excel(xls, sheet_name="Sources")
df_tests = pd.read_excel(xls, sheet_name="Test_Cases")

for col in ["match_id", "city", "stadium", "team_a", "team_b", "kickoff_time", "time_status"]:
    if col in df_matches.columns:
        df_matches[col] = df_matches[col].astype(str).str.strip()

for col in ["place_id", "name_ar", "name_en", "category", "walking_load", "family_friendly", "indoor_outdoor", "opening_hours"]:
    if col in df_places.columns:
        df_places[col] = df_places[col].astype(str).str.strip()

riyadh_matches = df_matches[df_matches["city"].str.casefold() == "riyadh".casefold()].copy()

required_match_cols = {"match_id", "date", "team_a", "team_b", "city", "stadium", "kickoff_time", "time_status"}
required_place_cols = {"place_id", "name_ar", "category", "latitude", "longitude", "indoor_outdoor", "visit_minutes", "walking_load", "family_friendly", "opening_hours"}
assert required_match_cols.issubset(df_matches.columns), "بعض أعمدة Matches_All غير موجودة"
assert required_place_cols.issubset(df_places.columns), "بعض أعمدة Places_Riyadh غير موجودة"
assert not riyadh_matches.empty, "لا توجد مباريات في Riyadh داخل الملف"
assert riyadh_matches["match_id"].is_unique, "يوجد تكرار في match_id لمباريات الرياض"

print(f"✅ الداتا جاهزة: {len(df_matches)} مباراة إجمالًا | {len(riyadh_matches)} مباراة في الرياض | {len(df_places)} وجهة بالرياض")
print("✅ Gemini API Key موجود" if GEMINI_API_KEY else "⚠️ Gemini API Key غير موجود — سيعمل Fallback إلى أن تضيفيه في Secrets")


In [ ]:

# 3) MASAR 27 — Premium UI + Gemini Function Calling + Structured Output + Deterministic Constraints
import gradio as gr
from google import genai
from urllib.parse import quote_plus

MODEL_NAME = "gemini-3.6-flash"

# إحداثيات تقريبية للنسخة التجريبية، تُستخدم فقط لحساب مسافات/أزمنة محافظة وروابط الخرائط.
# زمن الطريق هنا تقديري وليس بيانات مرور حيّة.
STADIUM_COORDS = {
    "King Fahd Sports City Stadium": (24.78855, 46.83914),
    "King Saud University Stadium": (24.72918, 46.62377),
    "Imam Mohammed Bin Saud University Stadium": (24.80631, 46.70350),
    "Al Shabab Stadium": (24.80311, 46.62879),
    "Kingdom Arena": (24.77868, 46.60596),
}

INTEREST_MAP = {
    "تراث وثقافة": ["heritage", "culture", "museum"],
    "فن": ["art", "culture", "museum", "technology"],
    "مقاهي": ["food", "dining", "viewpoint", "modern"],
    "تسوق": ["shopping", "modern"],
    "أماكن خارجية": ["nature", "outdoor", "park", "adventure", "leisure"],
    "مطاعم": ["dining", "food", "entertainment"],
}

MOBILITY_ALLOWED = {
    "منخفضة (مشي خفيف)": {"low"},
    "متوسطة (مشي عادي)": {"low", "medium"},
    "عالية (مشي مسافات)": {"low", "medium", "high"},
}

# النص الظاهر للمستخدم أبسط من القيم الداخلية المستخدمة في محرك القيود.
MOBILITY_UI_TO_INTERNAL = {
    "خفيف · مسافات قصيرة": "منخفضة (مشي خفيف)",
    "متوسط · مشي معتدل": "متوسطة (مشي عادي)",
    "مرن · أمشي أكثر": "عالية (مشي مسافات)",
}
GROUP_UI_TO_INTERNAL = {
    "لحالي": "فرد",
    "مع العائلة": "عائلة",
    "مع كبار السن": "كبار سن",
    "مع الأصدقاء": "أصدقاء",
}

WALKING_DISPLAY = {
    "low": "مشي خفيف",
    "medium": "مشي متوسط",
    "high": "مشي أكثر",
}
VENUE_DISPLAY = {
    "indoor": "داخلي",
    "outdoor": "خارجي",
    "mixed": "داخلي وخارجي",
}

CROWD_DISPLAY = {
    "low": "ازدحام متوقع: منخفض",
    "medium": "ازدحام متوقع: متوسط",
    "high": "ازدحام متوقع: مرتفع",
}

CONDITION_LABELS = {
    "طبيعية": "Normal",
    "حرارة مرتفعة": "High Heat",
    "ازدحام مرتفع": "High Traffic",
    "تأخرت 45 دقيقة": "Delayed 45 min",
    "أشعر بالتعب": "Fatigue",
    "وقت صلاة ضمن الجولة": "Prayer Time",
}

TRANSPORT_SPEED = {
    "سيارة": 35.0,
    "مترو": 30.0,
    "مشي": 4.5,
}

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "headline": {"type": "string"},
        "summary": {"type": "string"},
        "before_whistle_message": {"type": "string"},
        "itinerary": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "kind": {"type": "string", "enum": ["visit", "prayer", "stadium", "post_match"]},
                    "start_time": {"type": "string"},
                    "end_time": {"type": "string"},
                    "place_id": {"type": ["string", "null"]},
                    "title": {"type": "string"},
                    "reason": {"type": "string"},
                    "travel_minutes_to_next": {"type": "integer"},
                    "constraint_note": {"type": "string"},
                },
                "required": ["kind", "start_time", "end_time", "place_id", "title", "reason", "travel_minutes_to_next", "constraint_note"],
            },
        },
        "warnings": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["headline", "summary", "before_whistle_message", "itinerary", "warnings"],
}

TOOLS = [
    {
        "type": "function",
        "name": "get_match_info",
        "description": "Fetch the selected Riyadh match from the verified local dataset, including date, stadium and kickoff status.",
        "parameters": {
            "type": "object",
            "properties": {
                "match_id": {"type": "string", "description": "AFC match id such as AC27-M01"},
                "kickoff_override": {"type": "string", "description": "Optional user ticket kickoff in HH:MM. Empty string means use dataset provisional kickoff."},
            },
            "required": ["match_id", "kickoff_override"],
        },
    },
    {
        "type": "function",
        "name": "get_candidate_places",
        "description": "Filter and rank Riyadh attractions from the local dataset for both before-match and after-match planning using user interests, mobility, group, time and constraints.",
        "parameters": {
            "type": "object",
            "properties": {
                "match_id": {"type": "string"},
                "kickoff_override": {"type": "string"},
                "interests": {"type": "array", "items": {"type": "string"}},
                "mobility": {"type": "string"},
                "group_type": {"type": "string"},
                "group_size": {"type": "integer"},
                "available_from": {"type": "string"},
                "condition": {"type": "string"},
                "transport_mode": {"type": "string"},
            },
            "required": ["match_id", "kickoff_override", "interests", "mobility", "group_type", "group_size", "available_from", "condition", "transport_mode"],
        },
    },
    {
        "type": "function",
        "name": "get_prayer_times",
        "description": "Get Riyadh prayer times for the selected match date. This is used only to reserve a prayer break if it overlaps the tour window.",
        "parameters": {
            "type": "object",
            "properties": {"match_id": {"type": "string"}},
            "required": ["match_id"],
        },
    },
]


def _row_for_match(match_id):
    rows = riyadh_matches[riyadh_matches["match_id"] == str(match_id).strip()]
    if rows.empty:
        raise ValueError("المباراة غير موجودة ضمن مباريات الرياض في الـMVP")
    return rows.iloc[0]


def _parse_hhmm(value):
    s = str(value).strip()
    m = re.search(r"(\d{1,2}):(\d{2})", s)
    if not m:
        raise ValueError(f"وقت غير صالح: {value}")
    h, minute = int(m.group(1)), int(m.group(2))
    if h > 23 or minute > 59:
        raise ValueError(f"وقت غير صالح: {value}")
    return h, minute


def _match_datetime(row, kickoff_override=""):
    raw_date = pd.to_datetime(row["date"]).date()
    raw_time = kickoff_override.strip() if str(kickoff_override).strip() else str(row["kickoff_time"])
    h, m = _parse_hhmm(raw_time)
    return datetime(raw_date.year, raw_date.month, raw_date.day, h, m)


def _available_datetime(row, available_from, condition):
    d = pd.to_datetime(row["date"]).date()
    h, m = _parse_hhmm(available_from)
    dt = datetime(d.year, d.month, d.day, h, m)
    if condition == "تأخرت 45 دقيقة":
        dt += timedelta(minutes=45)
    return dt


def _fmt_ar(dt):
    if not isinstance(dt, datetime):
        return str(dt)
    h = dt.hour
    minute = dt.minute
    period = "ص" if h < 12 else "م"
    hh = h % 12 or 12
    return f"{hh}:{minute:02d} {period}"


def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def _estimate_travel_minutes(origin, destination, transport_mode, condition):
    if not origin or not destination:
        return 20
    km_air = _haversine_km(origin[0], origin[1], destination[0], destination[1])
    road_km = km_air * 1.30
    speed = TRANSPORT_SPEED.get(transport_mode, 35.0)
    if condition == "ازدحام مرتفع":
        speed = max(15.0, speed * 0.60)
    base = 8 if transport_mode == "سيارة" else (12 if transport_mode == "مترو" else 3)
    return max(5, int(math.ceil((road_km / max(speed, 1)) * 60 + base)))


def _known_open_at(opening_hours, when_dt, visit_minutes):
    s = str(opening_hours).strip()
    if s in {"LIVE_CHECK", "nan", "NaN", ""}:
        return True, "LIVE_CHECK"
    if s == "24/7":
        return True, "24/7"
    m = re.match(r"^(\d{2}):(\d{2})-(\d{2}):(\d{2})$", s)
    if not m:
        return True, "LIVE_CHECK"
    oh, om, ch, cm = map(int, m.groups())
    opening = when_dt.replace(hour=oh, minute=om, second=0, microsecond=0)
    closing = when_dt.replace(hour=ch, minute=cm, second=0, microsecond=0)
    if closing <= opening:
        closing += timedelta(days=1)
    end_visit = when_dt + timedelta(minutes=int(visit_minutes))
    return opening <= when_dt and end_visit <= closing, s


def get_match_info(match_id, kickoff_override=""):
    row = _row_for_match(match_id)
    kickoff = _match_datetime(row, kickoff_override)
    buffer_minutes = 120
    return {
        "match_id": str(row["match_id"]),
        "date": pd.to_datetime(row["date"]).strftime("%Y-%m-%d"),
        "team_a": str(row["team_a"]),
        "team_b": str(row["team_b"]),
        "city": str(row["city"]),
        "stadium": str(row["stadium"]),
        "kickoff": kickoff.strftime("%H:%M"),
        "kickoff_display": _fmt_ar(kickoff),
        "kickoff_status": "User Ticket" if kickoff_override.strip() else str(row["time_status"]),
        "safe_arrival_base": _fmt_ar(kickoff - timedelta(minutes=buffer_minutes)),
        "stadium_coordinates": STADIUM_COORDS.get(str(row["stadium"])),
    }


def _normalize_interests(interests):
    if isinstance(interests, (list, tuple, set)):
        selected = [str(x).strip() for x in interests if str(x).strip()]
    elif interests is None:
        selected = []
    else:
        selected = [str(interests).strip()] if str(interests).strip() else []
    return selected or ["تراث وثقافة"]


def _interest_score(category, interests):
    cat = str(category).casefold()
    selected = _normalize_interests(interests)
    score = 0
    for interest in selected:
        keywords = INTEREST_MAP.get(interest, [])
        matched = sum(1 for k in keywords if k in cat)
        if matched:
            # يكافئ المكان على تغطيته لاهتمام مختار، مع زيادة بسيطة عند تطابق أكثر من كلمة.
            score += 4 + max(0, matched - 1)
    return score

def _opening_note(opening_hours):
    value = str(opening_hours).strip()
    if value in {"LIVE_CHECK", "nan", "NaN", ""}:
        return "يفضّل التأكد من ساعات العمل قبل الزيارة."
    if value == "24/7":
        return "متاح على مدار اليوم حسب البيانات الحالية."
    return f"ساعات العمل المسجلة: {value}"


def _visit_reason(cand, interests, group_type, condition):
    parts = ["اخترناه لأنه يناسب اهتماماتك"]
    walk = str(cand.get("walking_load", "")).casefold()
    indoor = str(cand.get("indoor_outdoor", "")).casefold()
    crowd = str(cand.get("crowd_level_est", "")).casefold()
    if condition == "حرارة مرتفعة" and indoor == "indoor":
        parts.append("وهو مكان داخلي مناسب للأجواء الحارة")
    elif condition == "ازدحام مرتفع" and crowd == "low":
        parts.append("وازدحامه المتوقع أقل ضمن سيناريو اليوم")
    elif condition == "أشعر بالتعب" and walk == "low":
        parts.append("ويحتاج مشيًا خفيفًا")
    else:
        parts.append("ويناسب الوقت ومستوى المشي اللي اخترته")
    if group_type == "عائلة":
        parts.append("ومناسب للعائلة")
    elif group_type == "كبار سن":
        parts.append("وملائم أكثر لراحة كبار السن")
    return "، ".join(parts) + "."

def decrease_group_size(value):
    try:
        value = int(value or 1)
    except Exception:
        value = 1
    return max(1, value - 1)


def increase_group_size(value):
    try:
        value = int(value or 1)
    except Exception:
        value = 1
    return min(12, value + 1)


def _estimated_crowd_level(category, condition="طبيعية", phase="before"):
    """
    تقدير Prototype على مستوى كل محطة، وليس إشغالًا حيًا.
    يعتمد على نوع الوجهة وسيناريو اليوم ويُستخدم للترتيب عند الازدحام.
    """
    cat = str(category).casefold()
    score = 1
    popular_terms = ["shopping", "dining", "entertainment", "viewpoint", "family", "modern"]
    if any(term in cat for term in popular_terms):
        score = 2
    if "entertainment" in cat or "shopping & dining" in cat:
        score = 3
    if phase == "post" and any(term in cat for term in ["shopping", "dining", "entertainment", "viewpoint"]):
        score += 1
    if condition == "ازدحام مرتفع":
        score += 1
    score = max(1, min(3, score))
    return {1: "low", 2: "medium", 3: "high"}[score]


def get_candidate_places(match_id, kickoff_override, interests, mobility, group_type, group_size, available_from, condition, transport_mode):
    row = _row_for_match(match_id)
    kickoff = _match_datetime(row, kickoff_override)
    start = _available_datetime(row, available_from, condition)

    arrival_buffer = 150 if condition == "ازدحام مرتفع" else 120
    target_arrival = kickoff - timedelta(minutes=arrival_buffer)
    available_minutes = max(0, int((target_arrival - start).total_seconds() // 60))
    stadium_coord = STADIUM_COORDS.get(str(row["stadium"]))

    allowed_walk = set(MOBILITY_ALLOWED.get(mobility, {"low", "medium"}))
    if condition == "أشعر بالتعب" or group_type == "كبار سن":
        allowed_walk = {"low"}

    before_rows = []
    after_rows = []

    for _, p in df_places.iterrows():
        walk = str(p["walking_load"]).casefold()
        if walk not in allowed_walk:
            continue
        if group_type in {"عائلة", "كبار سن"} and str(p["family_friendly"]).casefold() != "yes":
            continue

        indoor = str(p["indoor_outdoor"]).casefold()
        if condition == "حرارة مرتفعة" and indoor == "outdoor":
            continue

        visit = int(float(p["visit_minutes"]))
        place_coord = (float(p["latitude"]), float(p["longitude"]))
        to_stadium = _estimate_travel_minutes(place_coord, stadium_coord, transport_mode, condition)
        from_stadium = _estimate_travel_minutes(stadium_coord, place_coord, transport_mode, condition)

        base_score = _interest_score(p["category"], interests)
        if condition == "حرارة مرتفعة" and indoor == "indoor":
            base_score += 5
        if walk == "low":
            base_score += 2 if mobility.startswith("منخفضة") or condition == "أشعر بالتعب" else 1
        if group_type == "عائلة" and str(p["family_friendly"]).casefold() == "yes":
            base_score += 2

        before_crowd = _estimated_crowd_level(p["category"], condition, "before")
        post_crowd = _estimated_crowd_level(p["category"], condition, "post")

        if condition == "ازدحام مرتفع":
            base_score -= {"low": 0.0, "medium": 1.5, "high": 4.0}[before_crowd]

        common = {
            "place_id": str(p["place_id"]),
            "name_ar": str(p["name_ar"]),
            "name_en": str(p.get("name_en", "")),
            "category": str(p["category"]),
            "latitude": float(p["latitude"]),
            "longitude": float(p["longitude"]),
            "indoor_outdoor": str(p["indoor_outdoor"]),
            "visit_minutes": visit,
            "walking_load": str(p["walking_load"]),
            "family_friendly": str(p["family_friendly"]),
            "opening_hours": str(p["opening_hours"]),
            "travel_minutes_to_stadium_est": to_stadium,
            "travel_minutes_from_stadium_est": from_stadium,
        }

        feasibility_need = visit + to_stadium + 25
        if feasibility_need <= available_minutes:
            known_open, opening_status = _known_open_at(p["opening_hours"], start + timedelta(minutes=15), visit)
            if known_open:
                item = dict(common)
                item["opening_status"] = opening_status
                item["crowd_level_est"] = before_crowd
                item["crowd_note"] = "تقدير للنموذج الأولي وليس بيانات إشغال حية."
                item["score"] = round(base_score - min(to_stadium / 20.0, 5), 2)
                before_rows.append(item)

        post_item = dict(common)
        post_item["opening_status"] = str(p["opening_hours"])
        post_item["crowd_level_est"] = post_crowd
        post_item["crowd_note"] = "تقدير للنموذج الأولي وليس بيانات إشغال حية."
        post_score = base_score - min(from_stadium / 25.0, 4)
        if condition == "ازدحام مرتفع":
            post_score -= {"low": 0.0, "medium": 1.0, "high": 3.0}[post_crowd]
        post_item["score"] = round(post_score, 2)
        after_rows.append(post_item)

    before_rows = sorted(before_rows, key=lambda x: (-x["score"], x["travel_minutes_to_stadium_est"]))[:10]
    after_rows = sorted(after_rows, key=lambda x: (-x["score"], x["travel_minutes_from_stadium_est"]))[:10]

    pairwise = {}
    for a in before_rows[:6]:
        for b in before_rows[:6]:
            if a["place_id"] == b["place_id"]:
                continue
            key = f"{a['place_id']}->{b['place_id']}"
            pairwise[key] = _estimate_travel_minutes(
                (a["latitude"], a["longitude"]),
                (b["latitude"], b["longitude"]),
                transport_mode,
                condition,
            )

    return {
        "effective_start": _fmt_ar(start),
        "target_stadium_arrival": _fmt_ar(target_arrival),
        "available_minutes_before_stadium": available_minutes,
        "condition": condition,
        "transport_mode": transport_mode,
        "mobility_filter": sorted(allowed_walk),
        "crowd_method": "per-stop prototype estimate; not live occupancy",
        "note": "Travel minutes are prototype estimates, not live traffic data.",
        "candidates": before_rows,
        "post_match_candidates": after_rows,
        "pairwise_travel_estimates": pairwise,
    }

def get_prayer_times(match_id):
    row = _row_for_match(match_id)
    date_obj = pd.to_datetime(row["date"]).date()
    date_str = date_obj.strftime("%d-%m-%Y")
    try:
        url = f"https://api.aladhan.com/v1/timingsByCity/{date_str}"
        params = {"city": "Riyadh", "country": "Saudi Arabia", "method": 4}
        r = requests.get(url, params=params, timeout=8)
        r.raise_for_status()
        t = r.json()["data"]["timings"]
        return {
            "status": "ok",
            "date": date_obj.isoformat(),
            "city": "Riyadh",
            "Fajr": t.get("Fajr"),
            "Dhuhr": t.get("Dhuhr"),
            "Asr": t.get("Asr"),
            "Maghrib": t.get("Maghrib"),
            "Isha": t.get("Isha"),
            "note": "Calculated prayer times for prototype planning; local verification recommended.",
        }
    except Exception as e:
        return {"status": "unavailable", "date": date_obj.isoformat(), "city": "Riyadh", "error": str(e)[:160]}


def _dispatch_function(name, args):
    if name == "get_match_info":
        return get_match_info(**args)
    if name == "get_candidate_places":
        return get_candidate_places(**args)
    if name == "get_prayer_times":
        return get_prayer_times(**args)
    return {"error": f"Unknown function: {name}"}


PRAYER_AR = {
    "Dhuhr": "الظهر",
    "Asr": "العصر",
    "Maghrib": "المغرب",
    "Isha": "العشاء",
}


def _prayer_slots(match_id, window_start, window_end):
    """Return prayer windows inside the tour window as concrete datetimes."""
    payload = get_prayer_times(match_id)
    slots = []
    if payload.get("status") != "ok":
        return slots
    for key in ["Dhuhr", "Asr", "Maghrib", "Isha"]:
        raw = payload.get(key)
        if not raw:
            continue
        try:
            h, m = _parse_hhmm(raw)
            pdt = window_start.replace(hour=h, minute=m, second=0, microsecond=0)
            if window_start <= pdt < window_end:
                slots.append((key, pdt))
        except Exception:
            pass
    return sorted(slots, key=lambda x: x[1])


def _fallback_plan(match_id, kickoff_override, interests, mobility, group_type, group_size, available_from, condition, transport_mode):
    """Fallback chooses candidates deterministically; hard times are rebuilt later by the constraint engine."""
    payload = get_candidate_places(
        match_id, kickoff_override, interests, mobility, group_type, group_size,
        available_from, condition, transport_mode
    )
    candidates = payload.get("candidates", [])
    draft = []
    for cand in candidates[:2]:
        draft.append({
            "kind": "visit",
            "start_time": "",
            "end_time": "",
            "place_id": cand["place_id"],
            "title": cand["name_ar"],
            "reason": f"اخترناه لأنه يناسب اهتماماتك ({'، '.join(_normalize_interests(interests))}) ويضبط مع وقتك قبل المباراة.",
            "travel_minutes_to_next": int(cand["travel_minutes_to_stadium_est"]),
            "constraint_note": _opening_note(cand["opening_hours"]),
        })

    return {
        "headline": "خطة يوم المباراة",
        "summary": "رتبنا يومك حول المباراة حسب وقتك واهتماماتك.",
        "before_whistle_message": "",
        "itinerary": draft,
        "warnings": [],
    }, ["fallback_rules"]


def _enforce_hard_constraints(plan, match_id, interests, mobility, group_type, group_size, available_from, condition, transport_mode):
    """
    Deterministic post-validation layer.
    Gemini chooses/ranks places; Python owns the clock so visits, prayer and stadium never overlap.
    """
    row = _row_for_match(match_id)
    kickoff = _match_datetime(row, "")
    arrival_buffer = 150 if condition == "ازدحام مرتفع" else 120
    safe_arrival = kickoff - timedelta(minutes=arrival_buffer)
    effective_start = _available_datetime(row, available_from, condition)

    candidates_payload = get_candidate_places(
        match_id, "", interests, mobility, group_type, group_size,
        available_from, condition, transport_mode
    )
    pre_candidates = candidates_payload.get("candidates", [])
    post_candidates = candidates_payload.get("post_match_candidates", [])
    pre_candidate_map = {c["place_id"]: c for c in pre_candidates}
    candidate_map = {c["place_id"]: c for c in (pre_candidates + post_candidates)}
    prayers = _prayer_slots(match_id, effective_start, kickoff)
    used_prayers = set()

    requested_visits = []
    seen_requested = set()
    for item in plan.get("itinerary", []):
        pid = item.get("place_id")
        if item.get("kind") == "visit" and pid in pre_candidate_map and pid not in seen_requested:
            requested_visits.append(item)
            seen_requested.add(pid)
    # If Gemini returned no valid candidate, keep the demo useful without inventing places.
    if not requested_visits:
        for cand in pre_candidates[:2]:
            requested_visits.append({
                "kind": "visit", "place_id": cand["place_id"], "title": cand["name_ar"],
                "reason": "خيار مناسب بعد مطابقة الوقت والاهتمامات ومستوى المشي.",
                "constraint_note": _opening_note(cand["opening_hours"]),
            })

    itinerary = []
    cursor = effective_start
    current_coord = None

    def add_due_prayer(projected_end):
        nonlocal cursor
        stadium_coord = STADIUM_COORDS.get(str(row["stadium"]))
        to_stadium_now = 20 if current_coord is None else _estimate_travel_minutes(current_coord, stadium_coord, transport_mode, condition)
        for prayer_name, prayer_dt in prayers:
            if prayer_name in used_prayers or prayer_dt >= safe_arrival:
                continue
            if cursor <= prayer_dt < projected_end:
                prayer_end = prayer_dt + timedelta(minutes=20)
                # Do not sacrifice the protected stadium deadline for a tourism stop.
                # If prayer is too close to safe arrival, it will be scheduled after arrival at the stadium.
                if prayer_end + timedelta(minutes=to_stadium_now) > safe_arrival:
                    return False
                itinerary.append({
                    "kind": "prayer",
                    "start_time": _fmt_ar(prayer_dt),
                    "end_time": _fmt_ar(prayer_end),
                    "place_id": None,
                    "title": f"صلاة {PRAYER_AR.get(prayer_name, '')}",
                    "reason": "خصصنا وقتًا للصلاة بدون ما يتداخل مع زيارتك.",
                    "travel_minutes_to_next": 0,
                    "constraint_note": "وقت الصلاة تقديري؛ يفضّل التحقق من الوقت المحلي.",
                    "_sort_dt": prayer_dt,
                })
                used_prayers.add(prayer_name)
                cursor = prayer_end
                return True
        return False

    for original in requested_visits[:3]:
        cand = pre_candidate_map.get(original.get("place_id"))
        if not cand:
            continue
        coord = (cand["latitude"], cand["longitude"])
        transfer = 15 if current_coord is None else _estimate_travel_minutes(current_coord, coord, transport_mode, condition)
        visit_minutes = int(cand["visit_minutes"])

        # If prayer would fall during transfer/visit, reserve it first and recalculate.
        projected_end = cursor + timedelta(minutes=transfer + visit_minutes)
        while add_due_prayer(projected_end):
            transfer = 15 if current_coord is None else _estimate_travel_minutes(current_coord, coord, transport_mode, condition)
            projected_end = cursor + timedelta(minutes=transfer + visit_minutes)

        start_visit = cursor + timedelta(minutes=transfer)
        end_visit = start_visit + timedelta(minutes=visit_minutes)
        to_stadium = int(cand["travel_minutes_to_stadium_est"])

        if end_visit + timedelta(minutes=to_stadium) > safe_arrival:
            continue

        opening_note = _opening_note(cand["opening_hours"])
        itinerary.append({
            "kind": "visit",
            "start_time": _fmt_ar(start_visit),
            "end_time": _fmt_ar(end_visit),
            "place_id": cand["place_id"],
            "title": cand["name_ar"],
            "reason": _visit_reason(cand, interests, group_type, condition),
            "travel_minutes_to_next": to_stadium,
            "constraint_note": opening_note,
            "crowd_level_est": cand.get("crowd_level_est"),
            "crowd_note": cand.get("crowd_note"),
            "_sort_dt": start_visit,
        })
        cursor = end_visit
        current_coord = coord

    # Reserve a remaining prayer only when it still preserves the safe-arrival deadline.
    current_to_stadium = 20
    if current_coord is not None:
        stadium_coord = STADIUM_COORDS.get(str(row["stadium"]))
        current_to_stadium = _estimate_travel_minutes(current_coord, stadium_coord, transport_mode, condition)
    for prayer_name, prayer_dt in prayers:
        if prayer_name in used_prayers or prayer_dt < cursor or prayer_dt >= safe_arrival:
            continue
        prayer_end = prayer_dt + timedelta(minutes=20)
        if prayer_end + timedelta(minutes=current_to_stadium) <= safe_arrival:
            itinerary.append({
                "kind": "prayer",
                "start_time": _fmt_ar(prayer_dt),
                "end_time": _fmt_ar(prayer_end),
                "place_id": None,
                "title": f"صلاة {PRAYER_AR.get(prayer_name, '')}",
                "reason": "خصصنا وقتًا للصلاة قبل ما تكمل طريقك للملعب.",
                "travel_minutes_to_next": current_to_stadium,
                "constraint_note": "وقت الصلاة تقديري؛ يفضّل التحقق من الوقت المحلي.",
                "_sort_dt": prayer_dt,
            })
            used_prayers.add(prayer_name)
            cursor = prayer_end
            break

    late = effective_start > safe_arrival
    stadium_arrival = effective_start if late else safe_arrival
    stadium_reason = (
        "الوقت صار ضيق، لذلك الأفضل تتجه للملعب مباشرة."
        if late else
        ("مع الازدحام، الأفضل تكون في الملعب قبل المباراة بساعتين ونصف عشان تدخل براحة."
         if condition == "ازدحام مرتفع"
         else "خلّنا نكون في الملعب قبل المباراة بساعتين عشان يكون دخولك أريح وبدون استعجال.")
    )
    stadium_note = (
        "إذا ضاق الوقت، نقلل المحطات قبل ما نأثر على وصولك للملعب."
        if not late else
        "التوجه للملعب الآن هو الخيار الأنسب."
    )
    itinerary.append({
        "kind": "stadium",
        "start_time": _fmt_ar(stadium_arrival),
        "end_time": _fmt_ar(kickoff),
        "place_id": None,
        "title": f"الوصول إلى {row['stadium']}",
        "reason": stadium_reason,
        "travel_minutes_to_next": 0,
        "constraint_note": stadium_note,
        "_sort_dt": stadium_arrival,
    })

    # If a prayer falls too close to the protected arrival deadline, schedule it after arrival at/near the stadium.
    for prayer_name, prayer_dt in prayers:
        if prayer_name in used_prayers:
            continue
        if prayer_dt < kickoff and prayer_dt + timedelta(minutes=20) > safe_arrival - timedelta(minutes=30):
            stadium_prayer_start = max(prayer_dt, stadium_arrival + timedelta(minutes=5))
            stadium_prayer_end = stadium_prayer_start + timedelta(minutes=20)
            if stadium_prayer_end < kickoff:
                itinerary.append({
                    "kind": "prayer",
                    "start_time": _fmt_ar(stadium_prayer_start),
                    "end_time": _fmt_ar(stadium_prayer_end),
                    "place_id": None,
                    "title": f"صلاة {PRAYER_AR.get(prayer_name, '')}",
                    "reason": "وقت الصلاة قريب من موعد الوصول للملعب، لذلك رتبناها بعد الوصول حتى ما تتأخر.",
                    "travel_minutes_to_next": 0,
                    "constraint_note": "رتبناها بحيث ما تأثر على وقت دخولك للملعب.",
                    "_sort_dt": stadium_prayer_start,
                })
                used_prayers.add(prayer_name)
                break

    # اقتراح واحد اختياري بعد المباراة، بشرط ألا نكرر مكانًا ظهر قبل المباراة.
    used_place_ids = {x.get("place_id") for x in itinerary if x.get("kind") == "visit" and x.get("place_id")}
    post_candidate = None
    for item in plan.get("itinerary", []):
        pid = item.get("place_id")
        if item.get("kind") == "post_match" and pid in candidate_map and pid not in used_place_ids:
            post_candidate = candidate_map[pid]
            break
    if post_candidate is None:
        for cand in post_candidates:
            if cand["place_id"] not in used_place_ids:
                post_candidate = cand
                break
    if post_candidate is not None:
        itinerary.append({
            "kind": "post_match",
            "start_time": "بعد نهاية المباراة",
            "end_time": "",
            "place_id": post_candidate["place_id"],
            "title": post_candidate["name_ar"],
            "reason": "بعد المباراة اخترنا لك تجربة ثانية تناسب اهتماماتك وتكمل يومك بدون تكرار الأماكن.",
            "travel_minutes_to_next": 0,
            "constraint_note": _opening_note(post_candidate["opening_hours"]),
            "crowd_level_est": post_candidate.get("crowd_level_est"),
            "crowd_note": post_candidate.get("crowd_note"),
            "_sort_dt": kickoff + timedelta(hours=2),
        })

    itinerary.sort(key=lambda x: x.get("_sort_dt", kickoff + timedelta(days=1)))
    for item in itinerary:
        item.pop("_sort_dt", None)

    visits_count = sum(1 for x in itinerary if x["kind"] == "visit")
    window_minutes = max(0, int((safe_arrival - effective_start).total_seconds() // 60))
    if late:
        before_msg = "الوقت صار ضيق، والأفضل تتجه للملعب مباشرة."
    elif visits_count == 0:
        before_msg = "الوقت المتاح قبل الوصول للملعب ما يكفي لزيارة مريحة؛ الأفضل تتجه للملعب مباشرة."
    elif visits_count == 1:
        before_msg = f"قبل المباراة نقدر نضيف لك تجربة واحدة من {_fmt_ar(effective_start)} إلى {_fmt_ar(safe_arrival)} بدون استعجال."
    else:
        before_msg = f"قبل المباراة نقدر نرتب لك {visits_count} تجارب من {_fmt_ar(effective_start)} إلى {_fmt_ar(safe_arrival)} ونخلي وصولك للملعب في وقته."

    standard_warnings = [
        "مواعيد بداية المباريات في النسخة الحالية مبدئية حتى يصدر الجدول النهائي.",
        "أوقات التنقل تقديرية وليست مبنية على حركة المرور المباشرة.",
        "ساعات عمل بعض الوجهات تتغير؛ تأكد منها قبل التوجه.",
        "مستويات الازدحام لكل محطة تقديرية في النموذج الأولي وليست بيانات إشغال حية.",
    ]
    return {
        "headline": plan.get("headline") or "مسارك التكيفي ليوم المباراة",
        "summary": plan.get("summary") or before_msg,
        "before_whistle_message": before_msg,
        "itinerary": itinerary,
        "warnings": standard_warnings,
    }


def _gemini_plan(match_id, kickoff_override, interests, mobility, group_type, group_size, available_from, condition, transport_mode):
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY is missing")

    client = genai.Client(api_key=GEMINI_API_KEY)
    function_args = {
        "match_id": match_id,
        "kickoff_override": kickoff_override or "",
        "interests": interests,
        "mobility": mobility,
        "group_type": group_type,
        "group_size": int(group_size),
        "available_from": available_from,
        "condition": condition,
        "transport_mode": transport_mode,
    }

    prompt = f"""
أنت محرك MASAR 27، مخطط سياحي تكيفي لزوار كأس آسيا السعودية 2027 في الرياض.
المستخدم اختار:
- match_id: {match_id}
- available_from: {available_from}
- interests: {interests} (قد تكون أكثر من قيمة)
- mobility: {mobility}
- group_type: {group_type}
- group_size: {int(group_size)}
- condition: {condition}
- transport_mode: {transport_mode}

قبل أن تخطط، استخدم الأدوات للوصول إلى بياناتنا بدل اختراع الحقائق:
1) استدع get_match_info.
2) استدع get_candidate_places.
3) استدع get_prayer_times.
لا تقترح أي مكان خارج candidate places التي تعيدها الدالة. راعِ جميع الاهتمامات المختارة ووازن بينها ضمن الوقت المتاح.
"""

    interaction = client.interactions.create(
        model=MODEL_NAME,
        input=prompt,
        tools=TOOLS,
        generation_config={
            "tool_choice": {
                "allowed_tools": {
                    "mode": "any",
                    "tools": ["get_match_info", "get_candidate_places", "get_prayer_times"],
                }
            }
        },
    )

    called = []
    for _ in range(4):
        calls = [s for s in interaction.steps if getattr(s, "type", None) == "function_call"]
        if not calls:
            break
        results = []
        for call in calls:
            args = dict(call.arguments or {})
            # نعيد فرض قيم الواجهة على الحقول الحساسة حتى لا يغيّرها النموذج.
            if call.name == "get_match_info":
                args = {"match_id": match_id, "kickoff_override": kickoff_override or ""}
            elif call.name == "get_candidate_places":
                args = function_args.copy()
            elif call.name == "get_prayer_times":
                args = {"match_id": match_id}
            result = _dispatch_function(call.name, args)
            called.append(call.name)
            results.append({
                "type": "function_result",
                "name": call.name,
                "call_id": call.id,
                "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
            })
        required_now = {"get_match_info", "get_candidate_places", "get_prayer_times"}
        missing = sorted(required_now - set(called))
        kwargs = dict(
            model=MODEL_NAME,
            previous_interaction_id=interaction.id,
            input=results,
            tools=TOOLS,
        )
        if missing:
            kwargs["generation_config"] = {
                "tool_choice": {"allowed_tools": {"mode": "any", "tools": missing}}
            }
        interaction = client.interactions.create(**kwargs)
        if not missing:
            break

    # إذا لم يستدعِ الثلاث أدوات لأي سبب، لا نخاطر بداتا مخترعة.
    required = {"get_match_info", "get_candidate_places", "get_prayer_times"}
    if not required.issubset(set(called)):
        raise RuntimeError(f"Function calling incomplete: {sorted(set(called))}")

    final_instruction = """
أنشئ الآن الخطة النهائية بالعربية فقط اعتمادًا على نتائج الأدوات السابقة.
قواعد إلزامية:
- موعد المباراة والوصول الآمن قيود صلبة لا تُكسر.
- لا تستخدم أي place_id لم تُعده get_candidate_places.
- إذا لم توجد نافذة آمنة، قل بوضوح إن الأفضل التوجه مباشرة للملعب.
- ضع الصلاة فقط إذا كان وقتها يقع داخل نافذة الجولة.
- لا تقدّم أزمنة التنقل كبيانات مرور حية؛ اذكر أنها تقديرية.
- LIVE_CHECK يعني ضرورة التحقق المباشر من ساعات العمل.
- عند High Heat فضّل Indoor، وعند Fatigue/Low Mobility فضّل المشي المنخفض.
- عند Delayed 45 min خطط من وقت البداية الفعلي الجديد، لا من الوقت الأصلي.
- عند High Traffic حافظ على هامش الوصول الأكبر الموجود في نتيجة get_candidate_places.
- اختر تجربة مناسبة بعد المباراة من post_match_candidates، ولا تكرر أي مكان استُخدم قبل المباراة، مع شرط التحقق من ساعات العمل.
- اختر الأماكن والترتيب والمنطق؛ محرك القيود في Python سيعيد حساب الأوقات نهائيًا لمنع أي تداخل.
"""

    final = client.interactions.create(
        model=MODEL_NAME,
        previous_interaction_id=interaction.id,
        input=final_instruction,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": PLAN_SCHEMA,
        },
    )

    plan = json.loads(final.output_text)
    return plan, called


def _gmaps_search(place_name="", city="Riyadh", lat=None, lon=None):
    """يفتح Google Maps باسم الوجهة بدل الإحداثيات الخام كلما توفر الاسم."""
    place_name = str(place_name or "").strip()
    if place_name:
        query = f"{place_name}, {city}, Saudi Arabia"
    elif lat is not None and lon is not None:
        query = f"{lat},{lon}"
    else:
        query = city
    return f"https://www.google.com/maps/search/?api=1&query={quote_plus(query)}"

def _gmaps_route(origin_name, stadium, transport_mode, city="Riyadh", lat=None, lon=None):
    """Directions مباشر من اسم الوجهة إلى اسم الملعب؛ الإحداثيات تستخدم كـfallback فقط."""
    origin_name = str(origin_name or "").strip()
    if origin_name:
        origin = f"{origin_name}, {city}, Saudi Arabia"
    elif lat is not None and lon is not None:
        origin = f"{lat},{lon}"
    else:
        origin = city
    destination = f"{stadium}, {city}, Saudi Arabia"
    travelmode = "driving" if transport_mode == "سيارة" else ("transit" if transport_mode == "مترو" else "walking")
    return (
        "https://www.google.com/maps/dir/?api=1"
        f"&origin={quote_plus(origin)}"
        f"&destination={quote_plus(destination)}"
        f"&travelmode={travelmode}&dir_action=navigate"
    )

def _place_by_id(place_id):
    if not place_id:
        return None
    rows = df_places[df_places["place_id"].astype(str) == str(place_id)]
    return None if rows.empty else rows.iloc[0]


def _replan_message(condition):
    messages = {
        "حرارة مرتفعة": "عدّلنا المسار عشان نقلل تعرضك للحرارة ونعطي أولوية أكبر للأماكن الداخلية.",
        "ازدحام مرتفع": "حدّثنا المسار حسب الازدحام: زدنا وقت الوصول للملعب وفضّلنا المحطات الأقل ازدحامًا قدر الإمكان.",
        "تأخرت 45 دقيقة": "بدأنا من وقتك الجديد واختصرنا المحطات إذا احتجنا، بدون ما نعرّض موعد المباراة للتأخير.",
        "أشعر بالتعب": "خففنا المشي والمحطات عشان يكون يومك أريح، مع الحفاظ على وقت الوصول للملعب.",
        "وقت صلاة ضمن الجولة": "أعدنا ترتيب المحطات حول وقت الصلاة بدون ما يتأثر موعد وصولك للملعب.",
    }
    return messages.get(condition, "حدّثنا خطتك حسب التغيير الجديد مع الحفاظ على موعد المباراة.")


def _render_html(plan, match, mode, transport_mode, condition, available_from, is_replan=False):
    mode_badge = "✦ مخطط بالذكاء الاصطناعي" if mode == "gemini" else "وضع تجريبي"
    mode_class = "ai" if mode == "gemini" else "fallback"

    row = _row_for_match(match["match_id"])
    kickoff_dt = _match_datetime(row, "")
    target_arrival = kickoff_dt - timedelta(minutes=150 if condition == "ازدحام مرتفع" else 120)
    effective_start = _available_datetime(row, available_from, condition)
    before_range = f"{_fmt_ar(effective_start)} — {_fmt_ar(target_arrival)}"
    has_post = any(x.get("kind") == "post_match" for x in plan.get("itinerary", []))

    if is_replan:
        hero_title = "خطة يومك بعد التعديل."
        eyebrow = "UPDATED MATCH DAY · MASAR 27"
        intro = _replan_message(condition)
        update_badge = "<div class='revision-badge'>تم التحديث</div>"
    else:
        hero_title = "خطة يومك جاهزة."
        eyebrow = "YOUR MATCH DAY · MASAR 27"
        intro = str(plan.get("before_whistle_message", ""))
        update_badge = ""

    if has_post:
        intro += " ورتبنا لك تجربة مختلفة بعد المباراة تكمل يومك." if is_replan else " وبعد المباراة أضفنا لك تجربة ثانية تكمل يومك حسب اختياراتك."

    cards = []
    phase_before_added = False
    phase_match_added = False
    phase_after_added = False

    for item in plan.get("itinerary", []):
        kind = item.get("kind", "visit")
        place = _place_by_id(item.get("place_id"))

        if kind in {"visit", "prayer"} and not phase_before_added and not phase_match_added:
            cards.append("<div class='phase-label'><span>01</span><div><strong>قبل المباراة</strong><small>نستفيد من وقتك بدون ما نستعجل الوصول</small></div></div>")
            phase_before_added = True
        if kind == "stadium" and not phase_match_added:
            cards.append("<div class='phase-label match-phase'><span>02</span><div><strong>المباراة</strong><small>نثبت وقت وصول مريح للملعب</small></div></div>")
            phase_match_added = True
        if kind == "post_match" and not phase_after_added:
            cards.append("<div class='phase-label after-phase'><span>03</span><div><strong>بعد المباراة</strong><small>نكمل التجربة بمكان مختلف يناسبك</small></div></div>")
            phase_after_added = True

        map_links = ""
        meta = ""
        if place is not None:
            lat, lon = float(place["latitude"]), float(place["longitude"])
            place_name = str(place["name_ar"])
            map_links = f"<a class='map-btn' target='_blank' href='{_gmaps_search(place_name, 'Riyadh', lat, lon)}'>افتح في الخرائط ↗</a>"
            if kind != "post_match":
                map_links += f"<a class='map-btn ghost' target='_blank' href='{_gmaps_route(place_name, match['stadium'], transport_mode, 'Riyadh', lat, lon)}'>الطريق للملعب ↗</a>"
            walk_label = WALKING_DISPLAY.get(str(place["walking_load"]).casefold(), "مشي متوسط")
            venue_label = VENUE_DISPLAY.get(str(place["indoor_outdoor"]).casefold(), str(place["indoor_outdoor"]))
            crowd_key = str(item.get("crowd_level_est") or "").casefold()
            crowd_label = CROWD_DISPLAY.get(crowd_key, "")
            crowd_chip = f"<span title='تقدير للنموذج الأولي وليس إشغالًا حيًا'>{html.escape(crowd_label)}</span>" if crowd_label else ""
            meta = (
                f"<div class='meta-row'><span>{html.escape(walk_label)}</span>"
                f"<span>{int(float(place['visit_minutes']))} دقيقة</span>"
                f"<span>{html.escape(venue_label)}</span>{crowd_chip}</div>"
            )

        if kind == "stadium":
            time_html = f"<div class='stadium-times'><div><small>خلّنا نكون في الملعب</small><strong>{html.escape(str(item.get('start_time','')))}</strong></div><div class='divider'></div><div><small>صافرة البداية</small><strong>{html.escape(str(item.get('end_time','')))}</strong></div></div>"
            cards.append(f"""
            <div class='timeline-card stadium-card'>
              <div class='timeline-dot stadium-dot'>⚽</div>
              <div class='card-kicker'>موعدك الأساسي</div>
              <h3>{html.escape(str(item.get('title','')))}</h3>
              {time_html}
              <p>{html.escape(str(item.get('reason','')))}</p>
              <div class='constraint'>{html.escape(str(item.get('constraint_note','')))}</div>
            </div>
            """)
            continue

        if kind == "post_match":
            time_label = "بعد نهاية المباراة"
            cls, dot, kicker = "post-card", "🌙", "تكملة يومك"
        elif kind == "prayer":
            time_label = f"{item.get('start_time','')} — {item.get('end_time','')}"
            cls, dot, kicker = "prayer-card", "◐", "وقت الصلاة"
        else:
            time_label = f"{item.get('start_time','')} — {item.get('end_time','')}"
            cls, dot, kicker = "visit-card", "◆", "تجربة مناسبة لك"

        cards.append(f"""
        <div class='timeline-card {cls}'>
          <div class='timeline-dot'>{dot}</div>
          <div class='card-head'><span class='card-kicker'>{kicker}</span><span class='time-chip'>{html.escape(str(time_label))}</span></div>
          <h3>{html.escape(str(item.get('title','')))}</h3>
          {meta}
          <p>{html.escape(str(item.get('reason','')))}</p>
          <div class='constraint'>{html.escape(str(item.get('constraint_note','')))}</div>
          <div class='card-actions'>{map_links}</div>
        </div>
        """)

    warning_html = "".join(f"<li>{html.escape(str(w))}</li>" for w in plan.get("warnings", []))
    arrival_note = "قبل المباراة بساعتين ونصف بسبب الازدحام" if condition == "ازدحام مرتفع" else "قبل المباراة بساعتين"
    timeline_heading = "خطة يومك بعد التعديل" if is_replan else "يومك مع مسار 27"
    return f"""
    <div class='masar-result' dir='rtl'>
      <section class='result-hero'>
        <div class='hero-badges'><div class='result-mode {mode_class}'>{mode_badge}</div>{update_badge}</div>
        <div class='result-eyebrow'>{eyebrow}</div>
        <h2>{hero_title}</h2>
        <p>{html.escape(intro)}</p>
        <div class='journey-pills'><span>قبل المباراة</span><b>←</b><span>المباراة</span><b>←</b><span>بعد المباراة</span></div>
      </section>

      <section class='match-summary'>
        <div class='match-main'>
          <span>مباراتك</span>
          <h3>{html.escape(match['team_a'])} <b>×</b> {html.escape(match['team_b'])}</h3>
          <p>{html.escape(match['stadium'])}</p>
        </div>
        <div class='summary-stat'><span>صافرة البداية</span><strong>{html.escape(match['kickoff_display'])}</strong><small>موعد مبدئي</small></div>
        <div class='summary-stat'><span>خلّنا نكون في الملعب</span><strong>{_fmt_ar(target_arrival)}</strong><small>{arrival_note}</small></div>
        <div class='summary-stat'><span>وقت تجربتك قبل المباراة</span><strong>{html.escape(before_range)}</strong><small>حسب وقت بداية يومك</small></div>
      </section>

      <div class='timeline-title'><span>{timeline_heading}</span><small>قبل المباراة وبعدها</small></div>
      <div class='timeline'>{''.join(cards)}</div>

      <details class='notes-box'>
        <summary>معلومات مهمة قبل تنطلق</summary>
        <ul>{warning_html}</ul>
      </details>
    </div>
    """

def build_plan(selected_label, available_from, interests, mobility, group_type, group_size, transport_mode, condition="طبيعية", is_replan=False):
    match_id = LABEL_TO_MATCH_ID[selected_label]
    kickoff_override = ""
    interests = _normalize_interests(interests)
    available_from = TIME_UI_TO_INTERNAL.get(available_from, available_from)
    if group_type == "لحالي":
        group_size = 1
    else:
        group_size = max(2, min(12, int(group_size or 2)))
    mobility = MOBILITY_UI_TO_INTERNAL.get(mobility, mobility)
    group_type = GROUP_UI_TO_INTERNAL.get(group_type, group_type)

    try:
        plan, called = _gemini_plan(match_id, kickoff_override, interests, mobility, group_type, group_size, available_from, condition, transport_mode)
        mode = "gemini"
        api_error = None
    except Exception as e:
        plan, called = _fallback_plan(match_id, kickoff_override, interests, mobility, group_type, group_size, available_from, condition, transport_mode)
        mode = "fallback"
        api_error = str(e)[:300]

    plan = _enforce_hard_constraints(
        plan, match_id, interests, mobility, group_type, group_size,
        available_from, condition, transport_mode
    )
    match = get_match_info(match_id, kickoff_override)
    rendered = _render_html(plan, match, mode, transport_mode, condition, available_from, is_replan=is_replan)
    tech = {
        "planning_mode": mode,
        "model": MODEL_NAME if mode == "gemini" else None,
        "gemini_api_key_detected": bool(GEMINI_API_KEY),
        "functions_called": sorted(set(called)),
        "structured_output": mode == "gemini",
        "constraint_engine": "deterministic Python post-validation (time, prayer, mobility, per-stop crowd estimate, match deadline)",
        "mobility_filtering": "walking_load hard filter; stricter for seniors/fatigue",
        "crowd_awareness": "per-stop prototype crowd estimate used in ranking; not live occupancy",
        "maps_integration": "Google Maps place-name search + named directions to stadium + prototype travel estimator",
        "adaptive_replanning": bool(is_replan),
        "replan_reason": condition if is_replan else None,
        "selected_match_id": match_id,
        "kickoff_status": match["kickoff_status"],
        "condition": condition,
        "api_error_if_fallback": api_error,
        "plan_json": plan,
    }
    return rendered, tech

def build_initial_plan(selected_label, available_from, interests, mobility, group_type, group_size, transport_mode):
    return build_plan(selected_label, available_from, interests, mobility, group_type, group_size, transport_mode, "طبيعية", is_replan=False)

def build_replanned_plan(selected_label, available_from, interests, mobility, group_type, group_size, transport_mode, condition):
    return build_plan(selected_label, available_from, interests, mobility, group_type, group_size, transport_mode, condition, is_replan=True)

# ---------- Match labels: Riyadh demo only ----------
AR_MONTHS = {1:"يناير",2:"فبراير",3:"مارس",4:"أبريل",5:"مايو",6:"يونيو",7:"يوليو",8:"أغسطس",9:"سبتمبر",10:"أكتوبر",11:"نوفمبر",12:"ديسمبر"}

def _clock_label(value):
    h, m = _parse_hhmm(value)
    period = "ص" if h < 12 else "م"
    hh = h % 12 or 12
    return f"{hh}:{m:02d} {period}"

LABEL_TO_MATCH_ID = {}
match_choices = []
for _, row in riyadh_matches.sort_values(["date", "kickoff_time"]).iterrows():
    d = pd.to_datetime(row["date"])
    date_label = f"{d.day:02d} {AR_MONTHS[d.month]}"
    label = f"{date_label}  ·  {row['team_a']} × {row['team_b']}  ·  {_clock_label(row['kickoff_time'])}"
    LABEL_TO_MATCH_ID[label] = str(row["match_id"])
    match_choices.append(label)

TIME_UI_TO_INTERNAL = {}
for h in range(8, 23):
    for m in (0, 30):
        internal = f"{h:02d}:{m:02d}"
        TIME_UI_TO_INTERNAL[_clock_label(internal)] = internal
TIME_CHOICES = list(TIME_UI_TO_INTERNAL.keys())


def _show_step(step_number):
    return tuple(gr.update(visible=(i == step_number)) for i in range(1, 5))


def _sync_group_size(group_label, current_size):
    if group_label == "لحالي":
        return gr.update(visible=False), 1
    try:
        size = int(current_size or 2)
    except Exception:
        size = 2
    return gr.update(visible=True), max(2, min(12, size))


def _review_html(selected_label, available_from, interests, transport_mode, mobility, group_label, group_size):
    match_id = LABEL_TO_MATCH_ID[selected_label]
    match = get_match_info(match_id, "")
    selected_interests = _normalize_interests(interests)
    interest_tags = "".join(f"<span>{html.escape(x)}</span>" for x in selected_interests)
    if group_label == "لحالي":
        people_text = "لحالك"
    else:
        try:
            size = max(2, int(group_size or 2))
        except Exception:
            size = 2
        people_text = f"{group_label} · {size} أشخاص"
    return f"""
    <div class='review-card' dir='rtl'>
      <div class='review-check'>✓</div>
      <div>
        <h3>تمام، الصورة اكتملت.</h3>
        <p>مباراتك <strong>{html.escape(match['team_a'])} × {html.escape(match['team_b'])}</strong> الساعة <strong>{html.escape(match['kickoff_display'])}</strong>، وتبدأ يومك <strong>{html.escape(available_from)}</strong>.</p>
        <div class='review-tags'>
          {interest_tags}<span>{html.escape(transport_mode)}</span><span>{html.escape(mobility)}</span><span>{html.escape(people_text)}</span>
        </div>
      </div>
    </div>
    """

def _go_review(selected_label, available_from, interests, transport_mode, mobility, group_label, group_size):
    s1, s2, s3, s4 = _show_step(4)
    return s1, s2, s3, s4, _review_html(selected_label, available_from, interests, transport_mode, mobility, group_label, group_size)

CSS = r"""
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans+Arabic:wght@400;500;600;700&display=swap');
:root,html,body,.gradio-container{
  color-scheme:light!important;
  --emerald:#075b42;--emerald2:#0a7654;--deep:#091f18;--gold:#c9a86a;--sand:#f4f1ea;--paper:#fffefa;--ink:#15231d;--muted:#6c776f;--line:#e3e1da;
  --body-background-fill:#f4f1ea!important;--body-background-fill-dark:#f4f1ea!important;
  --body-text-color:#15231d!important;--body-text-color-dark:#15231d!important;
  --block-background-fill:#fffefa!important;--block-background-fill-dark:#fffefa!important;
  --block-label-text-color:#273c33!important;--block-label-text-color-dark:#273c33!important;
  --input-background-fill:#ffffff!important;--input-background-fill-dark:#ffffff!important;
  --input-border-color:#deddd6!important;--input-border-color-dark:#deddd6!important;
}
html.dark,body.dark,.dark .gradio-container{color-scheme:light!important;background:#f4f1ea!important;color:#15231d!important}
.gradio-container input,.gradio-container textarea,.gradio-container select,.gradio-container [role="combobox"]{background:#fff!important;color:#15231d!important;-webkit-text-fill-color:#15231d!important}
.choice-cards label,.choice-cards span{color:#273c33!important}
#replan-shell,#tech-shell{background:#fbfaf6!important;color:#15231d!important}
#replan-shell summary,#tech-shell summary,#replan-shell label,#tech-shell label,#replan-shell p,#tech-shell p,#tech-shell pre,#tech-shell code{color:#15231d!important}
*{box-sizing:border-box}.gradio-container{max-width:1180px!important;margin:0 auto!important;padding:20px 26px 56px!important;background:radial-gradient(circle at 88% 2%,rgba(201,168,106,.12),transparent 23%),radial-gradient(circle at 4% 20%,rgba(7,91,66,.07),transparent 25%),var(--sand)!important;font-family:'IBM Plex Sans Arabic',system-ui,-apple-system,'Segoe UI',Tahoma,sans-serif!important}footer{opacity:.35}
.brandbar{direction:rtl;display:flex;align-items:center;justify-content:space-between;padding:6px 3px 16px;color:var(--deep)}.brand{display:flex;align-items:center;gap:11px}.brand-mark{width:42px;height:42px;border-radius:14px;background:var(--deep);color:#fff;display:grid;place-items:center;font-weight:800;box-shadow:0 8px 20px rgba(10,32,25,.15)}.brand-copy strong{display:block;font-size:1.05rem}.brand-copy small{color:var(--muted);font-size:.73rem}.demo-badge{font-size:.74rem;color:var(--emerald);background:#e8f0eb;border:1px solid #d8e5dc;border-radius:999px;padding:7px 11px;font-weight:700}
.hero-premium{direction:rtl;position:relative;overflow:hidden;border-radius:30px;padding:52px 54px 46px;background:linear-gradient(135deg,#081f18 0%,#074b37 58%,#0b7554 100%);box-shadow:0 24px 70px rgba(6,48,34,.18);margin-bottom:24px;color:white!important}.hero-premium:after{content:'27';position:absolute;left:24px;bottom:-52px;font-size:15rem;line-height:1;font-weight:800;color:rgba(255,255,255,.045);letter-spacing:-18px}.hero-premium .eyebrow{display:inline-flex;color:#e9d6ad;font-size:.76rem;font-weight:700;letter-spacing:1.4px;margin-bottom:16px}.hero-premium h1{color:#fff!important;margin:0;max-width:820px;font-size:2.55rem;line-height:1.35;font-weight:700}.hero-premium p{color:rgba(255,255,255,.84)!important;max-width:760px;margin:15px 0 0;font-size:1.02rem;line-height:1.9}.hero-line{width:52px;height:3px;background:var(--gold);border-radius:4px;margin-top:24px}
#planner-shell{direction:rtl;background:rgba(255,254,250,.96)!important;border:1px solid rgba(219,217,208,.92)!important;border-radius:28px!important;padding:30px!important;box-shadow:0 14px 42px rgba(28,48,39,.07)!important;margin-bottom:20px}.planner-heading{direction:rtl;margin-bottom:20px}.planner-heading span{color:var(--gold);font-size:.72rem;font-weight:800;letter-spacing:.8px}.planner-heading h2{color:var(--deep);font-size:1.5rem;margin:4px 0 5px}.planner-heading p{margin:0;color:var(--muted);font-size:.88rem}
.step-progress{display:flex;direction:rtl;align-items:center;gap:8px;margin:4px 0 24px}.step-progress i{height:3px;flex:1;background:#e6e2d9;border-radius:999px;overflow:hidden}.step-progress i.on{background:linear-gradient(90deg,var(--emerald),var(--emerald2))}.step-progress span{color:#8c7960;font-size:.72rem;font-weight:700;white-space:nowrap}.step-kicker{color:var(--gold);font-size:.72rem;font-weight:800;letter-spacing:.5px;margin-bottom:4px}.step-title{color:var(--deep);font-size:1.35rem;font-weight:800;margin:0 0 4px}.step-sub{color:var(--muted);font-size:.85rem;margin:0 0 22px;line-height:1.7}.question-label{direction:rtl;color:#273c33;font-size:.95rem;font-weight:700;margin:4px 2px 9px}.question-help{color:#7b857f;font-size:.74rem;margin:-3px 2px 9px}
.premium-input input,.premium-input button,.premium-input [role="combobox"]{min-height:52px!important;border-radius:14px!important}.choice-cards{direction:rtl!important}.choice-cards label{border:1px solid #deddd6!important;border-radius:14px!important;padding:11px 13px!important;background:#fff!important;transition:.15s ease!important}.choice-cards label:hover{border-color:#9bb8aa!important;background:#f7fbf8!important}.choice-cards input:checked+span{color:var(--emerald)!important;font-weight:700!important}.choice-cards label:has(input:checked){border-color:#7ca592!important;background:#edf5f0!important;box-shadow:0 0 0 2px rgba(7,91,66,.05)}.interest-cards label{min-height:48px!important;display:flex!important;align-items:center!important;justify-content:flex-start!important}.interest-cards input{accent-color:#075b42!important}.interest-cards span{font-size:.9rem!important}
.step-actions{gap:10px!important;margin-top:22px}.next-btn,.primary-btn{background:linear-gradient(135deg,#075b42,#0a7654)!important;color:white!important;border:0!important;border-radius:15px!important;min-height:52px!important;font-weight:700!important;font-size:.96rem!important;box-shadow:0 9px 20px rgba(7,91,66,.16)!important}.back-btn{background:#f3f1ea!important;color:#405047!important;border:1px solid #dfddd5!important;border-radius:15px!important;min-height:52px!important;font-weight:700!important}.number-field input{text-align:right!important;font-weight:700!important}.review-card{display:flex;gap:15px;align-items:flex-start;background:linear-gradient(135deg,#f4faf6,#fffdfa);border:1px solid #d8e6de;border-radius:20px;padding:20px;margin:4px 0 18px}.review-check{width:38px;height:38px;flex:0 0 38px;border-radius:50%;display:grid;place-items:center;background:#0a7654;color:white;font-weight:800}.review-card h3{margin:0 0 5px;color:var(--deep);font-size:1.12rem}.review-card p{margin:0;color:#53635a;line-height:1.75;font-size:.84rem}.review-tags{display:flex;flex-wrap:wrap;gap:6px;margin-top:11px}.review-tags span{background:#fff;border:1px solid #dfe7e2;border-radius:999px;padding:5px 9px;font-size:.69rem;color:#526259}
#replan-shell{direction:rtl;border:1px solid #deddd6!important;border-radius:18px!important;background:#fbfaf6!important;margin:16px 0 8px!important}.replan-btn{background:#132a21!important;color:white!important;border:0!important;border-radius:13px!important;font-weight:700!important;min-height:46px!important}
.masar-result{font-family:'IBM Plex Sans Arabic',system-ui,sans-serif;color:var(--ink);direction:rtl;margin-top:14px}.result-hero{position:relative;overflow:hidden;background:linear-gradient(125deg,#0b251c,#0a5d43);color:#fff;border-radius:25px;padding:30px 32px;box-shadow:0 16px 42px rgba(7,91,66,.14)}.result-eyebrow{color:#d8c294;font-size:.68rem;font-weight:800;letter-spacing:1.5px;margin-top:17px}.result-hero h2{color:#fff!important;font-size:1.9rem;margin:4px 0 7px}.result-hero p{color:rgba(255,255,255,.85)!important;margin:0;line-height:1.85;font-size:.93rem;max-width:850px}.result-mode{display:inline-flex;padding:6px 11px;border-radius:999px;font-size:.72rem;font-weight:800}.result-mode.ai{background:#e7f4ed;color:#075b42}.result-mode.fallback{background:#fff1cd;color:#7d5709}.hero-badges{display:flex;gap:7px;flex-wrap:wrap;align-items:center}.revision-badge{display:inline-flex;padding:6px 11px;border-radius:999px;font-size:.72rem;font-weight:800;background:#f3e7c9;color:#72551f;border:1px solid rgba(201,168,106,.38)}.journey-pills{display:flex;align-items:center;gap:8px;margin-top:18px;flex-wrap:wrap}.journey-pills span{background:rgba(255,255,255,.09);border:1px solid rgba(255,255,255,.15);padding:6px 10px;border-radius:999px;font-size:.7rem;color:#fff}.journey-pills b{font-weight:400;color:#d6c291}
.match-summary{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;margin:12px 0 28px}.match-main,.summary-stat{background:var(--paper);border:1px solid var(--line);border-radius:18px;padding:18px;box-shadow:0 4px 15px rgba(28,48,39,.035);min-width:0}.match-main>span,.summary-stat>span{display:block;color:var(--muted);font-size:.71rem;margin-bottom:7px}.match-main h3{margin:0;color:var(--deep);font-size:1.1rem}.match-main h3 b{color:var(--gold);font-weight:500;margin:0 4px}.match-main p{margin:5px 0 0;color:#637068;font-size:.77rem}.summary-stat strong{display:block;font-size:.98rem;color:var(--deep)}.summary-stat small{display:block;color:#8a7560;font-size:.67rem;margin-top:4px}
.timeline-title{display:flex;align-items:end;justify-content:space-between;margin:0 2px 14px}.timeline-title span{font-size:1.2rem;font-weight:800;color:var(--deep)}.timeline-title small{font-size:.72rem;color:var(--muted)}.timeline{position:relative;padding-right:28px}.timeline:before{content:'';position:absolute;right:8px;top:55px;bottom:20px;width:1px;background:linear-gradient(#0a7654,#d8d6ce 72%,transparent)}.phase-label{position:relative;display:flex;align-items:center;gap:10px;margin:22px 0 12px;padding:0 2px;color:var(--deep)}.phase-label:first-child{margin-top:0}.phase-label>span{width:30px;height:30px;border-radius:10px;background:#e7f1eb;color:var(--emerald);display:grid;place-items:center;font-size:.68rem;font-weight:800}.phase-label strong{display:block;font-size:.9rem}.phase-label small{display:block;color:var(--muted);font-size:.67rem;margin-top:1px}.match-phase>span{background:#0a7654;color:#fff}.after-phase>span{background:#eee7d9;color:#816c45}
.timeline-card{position:relative;background:var(--paper);border:1px solid var(--line);border-radius:20px;padding:20px 22px;margin-bottom:13px;box-shadow:0 7px 22px rgba(28,48,39,.045)}.timeline-dot{position:absolute;right:-29px;top:22px;width:18px;height:18px;border-radius:50%;display:grid;place-items:center;background:#fff;border:4px solid #0a7654;font-size:0;box-shadow:0 0 0 4px var(--sand)}.stadium-dot{font-size:.65rem;width:24px;height:24px;right:-32px;top:20px;border:0;background:#0a7654;color:white;box-shadow:0 0 0 5px var(--sand)}.card-head{display:flex;align-items:center;justify-content:space-between;gap:10px}.card-kicker{color:#8b7552;font-size:.68rem;font-weight:800}.time-chip{background:#f1f3ee;color:#33483d;border-radius:999px;padding:6px 10px;font-size:.74rem;font-weight:700}.timeline-card h3{color:#0a5d43!important;margin:10px 0 6px;font-size:1.09rem}.timeline-card p{color:#45564d;margin:8px 0;line-height:1.75;font-size:.86rem}.meta-row{display:flex;flex-wrap:wrap;gap:7px;margin:7px 0 4px}.meta-row span{background:#f6f5f0;border:1px solid #ece9df;border-radius:8px;padding:4px 8px;color:#6a746e;font-size:.69rem}.constraint{background:#f7f6f2;border-right:3px solid #d7c296;border-radius:10px;padding:9px 11px;color:#667068;font-size:.73rem;margin-top:11px}.card-actions{margin-top:12px}.map-btn{display:inline-block;text-decoration:none!important;background:#075b42;color:white!important;padding:8px 12px;border-radius:10px;font-size:.73rem;font-weight:700;margin-left:6px}.map-btn.ghost{background:#edf3ef;color:#075b42!important}.prayer-card{background:#fffcf3;border-color:#eee2bd}.post-card{background:#fcfbf7;border-color:#e6decd}.stadium-card{background:linear-gradient(135deg,#fbfffd,#f3faf6);border-color:#cfe1d7}.stadium-times{display:flex;align-items:center;gap:16px;background:#fff;border:1px solid #dce7e0;border-radius:13px;padding:12px 14px;margin:12px 0}.stadium-times div:not(.divider){display:flex;flex-direction:column;gap:2px}.stadium-times small{color:#728078;font-size:.67rem}.stadium-times strong{color:#0a5d43;font-size:.97rem}.stadium-times .divider{width:1px;height:34px;background:#e0e6e2}.notes-box{background:transparent;border:1px solid #d9d8d0;border-radius:14px;padding:12px 15px;color:#667069;font-size:.75rem;margin-top:14px}.notes-box summary{cursor:pointer;color:#46554d;font-weight:700}.notes-box ul{margin:10px 18px 0 0;line-height:1.8}
@media(max-width:900px){.hero-premium{padding:38px 30px}.hero-premium h1{font-size:2.05rem}.match-summary{grid-template-columns:1fr 1fr}}
@media(max-width:600px){.gradio-container{padding:12px!important}.brandbar{padding-top:2px}.demo-badge{display:none}.hero-premium{padding:30px 22px;border-radius:22px}.hero-premium h1{font-size:1.7rem}.hero-premium:after{font-size:9rem}#planner-shell{padding:20px!important}.match-summary{grid-template-columns:1fr}.timeline{padding-right:22px}.timeline-dot{right:-23px}.stadium-dot{right:-26px}.step-actions{flex-direction:column!important}.review-card{padding:16px}}
"""

with gr.Blocks(title="MASAR 27 | مسار 27") as demo:
    gr.HTML("""
    <div class='brandbar'>
      <div class='brand'>
        <div class='brand-mark'>27</div>
        <div class='brand-copy'><strong>MASAR 27 · مسار 27</strong><small>كأس آسيا السعودية 2027</small></div>
      </div>
      <div class='demo-badge'>النسخة التجريبية · الرياض</div>
    </div>
    <section class='hero-premium'>
      <div class='eyebrow'>SAUDI ARABIA · AFC ASIAN CUP 2027</div>
      <h1>من المباراة إلى المدينة… عِش التجربة كاملة.</h1>
      <p>مسار ذكي يرتّب وقتك قبل المباراة وبعدها، ويقترح لك تجارب تناسب اهتماماتك ووقتك.</p>
      <div class='hero-line'></div>
    </section>
    """)

    with gr.Column(elem_id="planner-shell"):
        gr.HTML("<div class='planner-heading'><span>PLAN YOUR MATCH DAY</span><h2>خلّنا نرتّب يومك</h2><p>أربع خطوات سريعة، والباقي على مسار 27.</p></div>")

        # STEP 1
        with gr.Column(visible=True) as step1:
            gr.HTML("<div class='step-progress'><span>1 من 4</span><i class='on'></i><i></i><i></i><i></i></div><div class='step-kicker'>الخطوة الأولى</div><div class='step-title'>نبدأ من المباراة</div><div class='step-sub'>نعرف موعدك الأساسي أول، وبعدها نبني اليوم حوله.</div>")
            gr.HTML("<div class='question-label'>أي مباراة رايح لها؟</div>")
            match_input = gr.Dropdown(match_choices, value=match_choices[0], show_label=False, container=False, elem_classes=["premium-input"])
            gr.HTML("<div class='question-label' style='margin-top:18px'>متى تحب تبدأ يومك؟</div><div class='question-help'>اختر الوقت اللي تكون فيه جاهز تبدأ التجربة.</div>")
            available_input = gr.Dropdown(TIME_CHOICES, value="11:00 ص", show_label=False, container=False, elem_classes=["premium-input"])
            with gr.Row(elem_classes=["step-actions"]):
                step1_next = gr.Button("التالي", elem_classes=["next-btn"])

        # STEP 2
        with gr.Column(visible=False) as step2:
            gr.HTML("<div class='step-progress'><span>2 من 4</span><i class='on'></i><i class='on'></i><i></i><i></i></div><div class='step-kicker'>الخطوة الثانية</div><div class='step-title'>خلّها على جوّك</div><div class='step-sub'>اختر اللي يهمك، وبعدها نضبط طريقة تنقلك خلال اليوم.</div>")
            gr.HTML("<div class='question-label'>وش تحب تكتشف؟</div><div class='question-help'>اختر اللي يهمك، وتقدر تختار أكثر من خيار.</div>")
            interests_input = gr.CheckboxGroup(
                list(INTEREST_MAP.keys()),
                value=["تراث وثقافة"],
                show_label=False,
                container=False,
                elem_classes=["choice-cards", "interest-cards"]
            )
            gr.HTML("<div class='question-label' style='margin-top:18px'>كيف بتتنقل اليوم؟</div>")
            transport_input = gr.Radio(["سيارة", "مترو", "مشي"], value="سيارة", show_label=False, container=False, elem_classes=["choice-cards"])
            with gr.Row(elem_classes=["step-actions"]):
                step2_back = gr.Button("رجوع", elem_classes=["back-btn"])
                step2_next = gr.Button("التالي", elem_classes=["next-btn"])

        # STEP 3
        with gr.Column(visible=False) as step3:
            gr.HTML("<div class='step-progress'><span>3 من 4</span><i class='on'></i><i class='on'></i><i class='on'></i><i></i></div><div class='step-kicker'>الخطوة الثالثة</div><div class='step-title'>نضبطها على راحتك</div><div class='step-sub'>هذي التفاصيل تساعدنا نختار أماكن تناسب يومك فعلًا.</div>")
            gr.HTML("<div class='question-label'>وش مستوى المشي اللي يناسبك؟</div>")
            mobility_input = gr.Radio(list(MOBILITY_UI_TO_INTERNAL.keys()), value="متوسط · مشي معتدل", show_label=False, container=False, elem_classes=["choice-cards"])
            gr.HTML("<div class='question-label' style='margin-top:18px'>مين معك؟</div>")
            group_input = gr.Radio(list(GROUP_UI_TO_INTERNAL.keys()), value="لحالي", show_label=False, container=False, elem_classes=["choice-cards"])
            with gr.Column(visible=False) as group_size_wrap:
                gr.HTML("<div class='question-label' style='margin-top:18px'>كم عددكم؟</div>")
                group_size_input = gr.Number(value=1, minimum=1, maximum=12, step=1, precision=0, show_label=False, container=False, elem_classes=["premium-input","number-field"])
            with gr.Row(elem_classes=["step-actions"]):
                step3_back = gr.Button("رجوع", elem_classes=["back-btn"])
                step3_next = gr.Button("راجع اختياراتي", elem_classes=["next-btn"])

        # STEP 4
        with gr.Column(visible=False) as step4:
            gr.HTML("<div class='step-progress'><span>4 من 4</span><i class='on'></i><i class='on'></i><i class='on'></i><i class='on'></i></div><div class='step-kicker'>الخطوة الأخيرة</div><div class='step-title'>جاهزين</div><div class='step-sub'>راجع اختياراتك بسرعة، وبعدها نرتّب يوم المباراة كامل.</div>")
            review_html = gr.HTML(container=False)
            with gr.Row(elem_classes=["step-actions"]):
                step4_back = gr.Button("تعديل", elem_classes=["back-btn"])
                create_btn = gr.Button("رتّب لي يوم المباراة  ✦", variant="primary", elem_classes=["primary-btn"])

    output_html = gr.HTML(container=False)

    with gr.Accordion("تغيّر شيء في يومك؟", open=False, elem_id="replan-shell"):
        gr.Markdown("لو تغيّرت الظروف، حدّثها هنا. مسار 27 يعيد ترتيب اليوم ويقول لك وش تغيّر، مع الحفاظ على موعد المباراة.")
        condition_input = gr.Radio(
            ["حرارة مرتفعة", "ازدحام مرتفع", "تأخرت 45 دقيقة", "أشعر بالتعب", "وقت صلاة ضمن الجولة"],
            value="ازدحام مرتفع",
            label="وش تغيّر؟",
            elem_classes=["choice-cards", "condition-cards"]
        )
        replan_btn = gr.Button("رتّب يومي من جديد", elem_classes=["replan-btn"])

    with gr.Accordion("إثبات القدرات التقنية — للمحكّمين", open=False, elem_id="tech-shell"):
        tech_json = gr.JSON(label="Technical trace")

    # Conversational navigation
    step1_next.click(lambda: _show_step(2), outputs=[step1, step2, step3, step4])
    step2_back.click(lambda: _show_step(1), outputs=[step1, step2, step3, step4])
    step2_next.click(lambda: _show_step(3), outputs=[step1, step2, step3, step4])
    step3_back.click(lambda: _show_step(2), outputs=[step1, step2, step3, step4])
    step4_back.click(lambda: _show_step(3), outputs=[step1, step2, step3, step4])

    group_input.change(
        _sync_group_size,
        inputs=[group_input, group_size_input],
        outputs=[group_size_wrap, group_size_input]
    )

    step3_next.click(
        _go_review,
        inputs=[match_input, available_input, interests_input, transport_input, mobility_input, group_input, group_size_input],
        outputs=[step1, step2, step3, step4, review_html]
    )

    base_inputs = [match_input, available_input, interests_input, mobility_input, group_input, group_size_input, transport_input]
    create_btn.click(build_initial_plan, inputs=base_inputs, outputs=[output_html, tech_json])
    replan_btn.click(build_replanned_plan, inputs=base_inputs + [condition_input], outputs=[output_html, tech_json])

demo.launch(
    share=True,
    inline=True,
    theme=gr.themes.Soft(),
    css=CSS,
    show_error=True,
)


## اختبار التسليم النهائي

1. `Run all` يجب أن يجهّز `MASAR27_DATA.xlsx` تلقائيًا بدون Upload يدوي.
2. تأكدي أن `GEMINI_API_KEY` موجود في Colab Secrets وأن النتيجة تعرض «مخطط بالذكاء الاصطناعي».
3. اختبري حالة طبيعية، حرارة مرتفعة، وتأخير 45 دقيقة.
4. بعد «رتّب يومي من جديد» يجب أن يظهر بوضوح «خطة يومك بعد التعديل» وسبب التغيير.
5. افتحي «افتح في الخرائط»: يجب أن يبحث Google Maps باسم المكان، لا بالإحداثيات فقط.
6. جرّبي «الطريق للملعب»: يجب أن يفتح Directions من اسم الوجهة إلى اسم الملعب.
7. بدّلي الجهاز إلى Dark Mode: يجب أن تبقى واجهة مسار 27 بنفس الهوية الفاتحة والنصوص واضحة.
8. افتحي «إثبات القدرات التقنية — للمحكّمين» وتحققي من `mobility_filtering` و`crowd_awareness` و`adaptive_replanning`.
